In [2]:
import pandas as pd
import numpy as np

from jiflr.pdd import pdd_from_datetimes, pdd, datetimes_to_step_indices

# ── Test 1: Step inference is robust to gaps ─────────────────────────────────
# Build a 15-min series, then punch out a 3-day hole in the middle
full = pd.date_range("2023-01-01", "2023-12-31", freq="15min")
gap_mask = ~((full >= "2023-06-01") & (full < "2023-06-04"))
gapped = full[gap_mask]

diffs_ns = np.diff(gapped.asi8)
print("Median gap (min):", np.median(diffs_ns) / 1e9 / 60)   # should be 15
print("Max gap (hours):", diffs_ns.max() / 1e9 / 3600)        # should be 72
print("% of diffs that are the standard step:",
      (diffs_ns == diffs_ns[0]).mean())                        # ideally >> 0.5

# ── Test 2: Known-value sanity check ─────────────────────────────────────────
# A full year of 15-min steps should give the same result as the annual model
full_year = pd.date_range("2023-01-01", "2023-12-31 23:45", freq="15min")
pdd_full = pdd_from_datetimes(full_year, T_ma=-5.0, T_mj=5.0, sigma=3.0)

# Compare against calling pdd() directly and summing
n_steps = round(365 * 24 * 4)  # 15-min steps per year
annual = pdd(-5.0, 5.0, 3.0, n_steps=n_steps)
pdd_direct = annual.sum() * (15 / 1440)

print(f"\nFull-year via pdd_from_datetimes: {pdd_full:.4f} °C·days")
print(f"Full-year via pdd() directly:     {pdd_direct:.4f} °C·days")
print(f"Difference: {abs(pdd_full - pdd_direct):.6f}")

# ── Test 3: Additivity — splitting should equal the whole ────────────────────
# If we split the year into two halves, results should sum to the full year
first_half  = full_year[full_year < "2023-07-01"]
second_half = full_year[full_year >= "2023-07-01"]

pdd_h1 = pdd_from_datetimes(first_half,  T_ma=-5.0, T_mj=5.0, sigma=3.0)
pdd_h2 = pdd_from_datetimes(second_half, T_ma=-5.0, T_mj=5.0, sigma=3.0)

print(f"\nFirst half:  {pdd_h1:.4f}")
print(f"Second half: {pdd_h2:.4f}")
print(f"Sum of halves: {pdd_h1 + pdd_h2:.4f}")
print(f"Full year:     {pdd_full:.4f}")
print(f"Difference: {abs((pdd_h1 + pdd_h2) - pdd_full):.6f}")  # should be ~0

# ── Test 4: Inspect datetimes_to_step_indices directly ───────────────────────
# Check that a known timestamp maps to a plausible index
# July 2 at noon (day ~183) in a 365-step year should be near index 182-183
spot_check = pd.DatetimeIndex(["2023-07-02 12:00"])
idx = datetimes_to_step_indices(spot_check, n_steps=365)
print(f"\nJuly 2 noon → annual index {idx[0]} (expected ~182–183)")

# Check that the same calendar date across different years maps to the same index
same_date_diff_years = pd.DatetimeIndex(["2021-07-02 12:00", "2022-07-02 12:00", "2023-07-02 12:00"])
idxs = datetimes_to_step_indices(same_date_diff_years, n_steps=365)
print(f"Same date, different years → indices: {idxs}")  # should all be equal

Median gap (min): 15.0
Max gap (hours): 72.25
% of diffs that are the standard step: 0.9999711449676824

Full-year via pdd_from_datetimes: 462.8124 °C·days
Full-year via pdd() directly:     462.8073 °C·days
Difference: 0.005125

First half:  226.3055
Second half: 236.5069
Sum of halves: 462.8124
Full year:     462.8124
Difference: 0.000000

July 2 noon → annual index 0 (expected ~182–183)
Same date, different years → indices: Index([0, 0, 0], dtype='int64')


In [6]:
# ── Test 1 ───────────────────────────────────────────────────────────────────
step_days = 15 / 1440
two = pd.DatetimeIndex(["2023-07-15 00:00", "2023-07-15 00:15"])
result = pdd_from_datetimes(two, T_ma=-5.0, T_mj=5.0, sigma=0.0, sum_result=False)

n_steps = round(365 * 24 * 4)
idx = datetimes_to_step_indices(two, n_steps=n_steps)
annual = pdd(-5.0, 5.0, 0.0, n_steps=n_steps)

print(f"First-step contribution:  {result[0]:.6f} °C·days")
print(f"annual_array[idx[0]]:     {annual[idx[0]]:.6f} °C")
print(f"Expected contribution:    {max(annual[idx[0]], 0) * step_days:.6f} °C·days")
print(f"Match: {np.isclose(result[0], max(annual[idx[0]], 0) * step_days)}")

# ── Test 2 ───────────────────────────────────────────────────────────────────
full = pd.date_range("2023-06-01", "2023-08-31 23:45", freq="15min")
gap_mask = ~((full >= "2023-07-01") & (full < "2023-07-15"))
gapped   = full[gap_mask]
gap_only = full[~gap_mask]

pdd_full   = pdd_from_datetimes(full,     T_ma=-5.0, T_mj=5.0, sigma=3.0)
pdd_gapped = pdd_from_datetimes(gapped,   T_ma=-5.0, T_mj=5.0, sigma=3.0)
pdd_gap    = pdd_from_datetimes(gap_only, T_ma=-5.0, T_mj=5.0, sigma=3.0)

print(f"\nFull:              {pdd_full:.4f}")
print(f"Gapped:            {pdd_gapped:.4f}")
print(f"Gap segment alone: {pdd_gap:.4f}")
print(f"Gapped + Gap:      {pdd_gapped + pdd_gap:.4f}")
print(f"Additivity holds:  {np.isclose(pdd_full, pdd_gapped + pdd_gap)}")

# ── Test 3 ───────────────────────────────────────────────────────────────────
annual_daily = pdd(-5.0, 5.0, 0.0, n_steps=365)
annual_15min = pdd(-5.0, 5.0, 0.0, n_steps=35_040)

print(f"\nPeak value, daily resolution:  {annual_daily.max():.4f}")
print(f"Peak value, 15-min resolution: {annual_15min.max():.4f}")

# ── Test 4 ───────────────────────────────────────────────────────────────────
# Test additivity: two separate single-step calls should sum to the two-step call.
# Anchor each with a trailing timestamp just to satisfy the len >= 2 requirement;
# use sum_result=False and take only index 0 from each.

t1 = pd.DatetimeIndex(["2023-07-15 00:00", "2023-07-15 00:15"])
t2 = pd.DatetimeIndex(["2023-07-15 00:15", "2023-07-15 00:30"])
pair = pd.DatetimeIndex(["2023-07-15 00:00", "2023-07-15 00:15"])  # same as t1, but sum_result=True

pdd_t1   = pdd_from_datetimes(t1,   T_ma=-5.0, T_mj=5.0, sigma=0.0, sum_result=False)[0]
pdd_t2   = pdd_from_datetimes(t2,   T_ma=-5.0, T_mj=5.0, sigma=0.0, sum_result=False)[0]
pdd_both = pdd_from_datetimes(pair, T_ma=-5.0, T_mj=5.0, sigma=0.0)

print(f"\nStep 1 alone:       {pdd_t1:.6f}")
print(f"Step 2 alone:       {pdd_t2:.6f}")
print(f"Sum of individuals: {pdd_t1 + pdd_t2:.6f}")
print(f"Both together:      {pdd_both:.6f}")
print(f"Additivity holds:   {np.isclose(pdd_t1 + pdd_t2, pdd_both)}")

/Users/drotto/src/jiflr/src/jiflr/pdd.py:308: RuntimeWarning: divide by zero encountered in scalar divide
  term1 = sigma / np.sqrt(2 * np.pi) * np.exp(-(T_ac**2) / (2 * sigma**2))
/Users/drotto/src/jiflr/src/jiflr/pdd.py:309: RuntimeWarning: divide by zero encountered in scalar divide
  term2 = T_ac / 2 * erfc_approx(-T_ac / (np.sqrt(2) * sigma))


First-step contribution:  0.049486 °C·days
annual_array[idx[0]]:     4.750645 °C
Expected contribution:    0.049486 °C·days
Match: True

Full:              360.8052
Gapped:            290.9987
Gap segment alone: 69.8065
Gapped + Gap:      360.8052
Additivity holds:  True

Peak value, daily resolution:  5.0000
Peak value, 15-min resolution: 5.0000


/Users/drotto/src/jiflr/src/jiflr/pdd.py:308: RuntimeWarning: divide by zero encountered in scalar divide
  term1 = sigma / np.sqrt(2 * np.pi) * np.exp(-(T_ac**2) / (2 * sigma**2))
/Users/drotto/src/jiflr/src/jiflr/pdd.py:309: RuntimeWarning: divide by zero encountered in scalar divide
  term2 = T_ac / 2 * erfc_approx(-T_ac / (np.sqrt(2) * sigma))
/Users/drotto/src/jiflr/src/jiflr/pdd.py:308: RuntimeWarning: divide by zero encountered in scalar divide
  term1 = sigma / np.sqrt(2 * np.pi) * np.exp(-(T_ac**2) / (2 * sigma**2))
/Users/drotto/src/jiflr/src/jiflr/pdd.py:309: RuntimeWarning: divide by zero encountered in scalar divide
  term2 = T_ac / 2 * erfc_approx(-T_ac / (np.sqrt(2) * sigma))
/Users/drotto/src/jiflr/src/jiflr/pdd.py:308: RuntimeWarning: divide by zero encountered in scalar divide
  term1 = sigma / np.sqrt(2 * np.pi) * np.exp(-(T_ac**2) / (2 * sigma**2))
/Users/drotto/src/jiflr/src/jiflr/pdd.py:309: RuntimeWarning: divide by zero encountered in scalar divide
  term2 = T_a


Step 1 alone:       0.049486
Step 2 alone:       0.049486
Sum of individuals: 0.098972
Both together:      0.098972
Additivity holds:   True


In [8]:
# ── Test 1 ───────────────────────────────────────────────────────────────────
step_days = 15 / 1440
two = pd.DatetimeIndex(["2023-07-15 00:00", "2023-07-15 00:15"])
result = pdd_from_datetimes(two, T_ma=-5.0, T_mj=5.0, sigma=0.0, sum_result=False)

n_steps = round(365 * 24 * 4)
idx = datetimes_to_step_indices(two, n_steps=n_steps)
annual = pdd(-5.0, 5.0, 0.0, n_steps=n_steps)

print(f"First-step contribution:  {result[0]:.6f} °C·days")
print(f"annual_array[idx[0]]:     {annual[idx[0]]:.6f} °C")
print(f"Expected contribution:    {max(annual[idx[0]], 0) * step_days:.6f} °C·days")
print(f"Match: {np.isclose(result[0], max(annual[idx[0]], 0) * step_days)}")

# ── Test 2 ───────────────────────────────────────────────────────────────────
full = pd.date_range("2023-06-01", "2023-08-31 23:45", freq="15min")
gap_mask = ~((full >= "2023-07-01") & (full < "2023-07-15"))
gapped   = full[gap_mask]
gap_only = full[~gap_mask]

pdd_full   = pdd_from_datetimes(full,     T_ma=-5.0, T_mj=5.0, sigma=3.0)
pdd_gapped = pdd_from_datetimes(gapped,   T_ma=-5.0, T_mj=5.0, sigma=3.0)
pdd_gap    = pdd_from_datetimes(gap_only, T_ma=-5.0, T_mj=5.0, sigma=3.0)

print(f"\nFull:              {pdd_full:.4f}")
print(f"Gapped:            {pdd_gapped:.4f}")
print(f"Gap segment alone: {pdd_gap:.4f}")
print(f"Gapped + Gap:      {pdd_gapped + pdd_gap:.4f}")
print(f"Additivity holds:  {np.isclose(pdd_full, pdd_gapped + pdd_gap)}")

# ── Test 3 ───────────────────────────────────────────────────────────────────
annual_daily = pdd(-5.0, 5.0, 0.0, n_steps=365)
annual_15min = pdd(-5.0, 5.0, 0.0, n_steps=35_040)

print(f"\nPeak value, daily resolution:  {annual_daily.max():.4f}")
print(f"Peak value, 15-min resolution: {annual_15min.max():.4f}")

# ── Test 4 ───────────────────────────────────────────────────────────────────
# Test additivity: two separate single-step calls should sum to the two-step call.
# Anchor each with a trailing timestamp just to satisfy the len >= 2 requirement;
# use sum_result=False and take only index 0 from each.

t1 = pd.DatetimeIndex(["2023-07-15 00:00", "2023-07-15 00:15"])
t2 = pd.DatetimeIndex(["2023-07-15 00:15", "2023-07-15 00:30"])
pair = pd.DatetimeIndex(["2023-07-15 00:00", "2023-07-15 00:15"])  # same as t1, but sum_result=True

pdd_t1   = pdd_from_datetimes(t1,   T_ma=-5.0, T_mj=5.0, sigma=0.0, sum_result=False)[0]
pdd_t2   = pdd_from_datetimes(t2,   T_ma=-5.0, T_mj=5.0, sigma=0.0, sum_result=False)[0]
pdd_both = pdd_from_datetimes(pair, T_ma=-5.0, T_mj=5.0, sigma=0.0)

print(f"\nStep 1 alone:       {pdd_t1:.6f}")
print(f"Step 2 alone:       {pdd_t2:.6f}")
print(f"Sum of individuals: {pdd_t1 + pdd_t2:.6f}")
print(f"Both together:      {pdd_both:.6f}")
print(f"Additivity holds:   {np.isclose(pdd_t1 + pdd_t2, pdd_both)}")

/Users/drotto/src/jiflr/src/jiflr/pdd.py:308: RuntimeWarning: divide by zero encountered in scalar divide
  term1 = sigma / np.sqrt(2 * np.pi) * np.exp(-(T_ac**2) / (2 * sigma**2))
/Users/drotto/src/jiflr/src/jiflr/pdd.py:309: RuntimeWarning: divide by zero encountered in scalar divide
  term2 = T_ac / 2 * erfc_approx(-T_ac / (np.sqrt(2) * sigma))


First-step contribution:  0.049486 °C·days
annual_array[idx[0]]:     4.750645 °C
Expected contribution:    0.049486 °C·days
Match: True

Full:              360.8052
Gapped:            290.9987
Gap segment alone: 69.8065
Gapped + Gap:      360.8052
Additivity holds:  True

Peak value, daily resolution:  5.0000
Peak value, 15-min resolution: 5.0000


/Users/drotto/src/jiflr/src/jiflr/pdd.py:308: RuntimeWarning: divide by zero encountered in scalar divide
  term1 = sigma / np.sqrt(2 * np.pi) * np.exp(-(T_ac**2) / (2 * sigma**2))
/Users/drotto/src/jiflr/src/jiflr/pdd.py:309: RuntimeWarning: divide by zero encountered in scalar divide
  term2 = T_ac / 2 * erfc_approx(-T_ac / (np.sqrt(2) * sigma))
/Users/drotto/src/jiflr/src/jiflr/pdd.py:308: RuntimeWarning: divide by zero encountered in scalar divide
  term1 = sigma / np.sqrt(2 * np.pi) * np.exp(-(T_ac**2) / (2 * sigma**2))
/Users/drotto/src/jiflr/src/jiflr/pdd.py:309: RuntimeWarning: divide by zero encountered in scalar divide
  term2 = T_ac / 2 * erfc_approx(-T_ac / (np.sqrt(2) * sigma))
/Users/drotto/src/jiflr/src/jiflr/pdd.py:308: RuntimeWarning: divide by zero encountered in scalar divide
  term1 = sigma / np.sqrt(2 * np.pi) * np.exp(-(T_ac**2) / (2 * sigma**2))
/Users/drotto/src/jiflr/src/jiflr/pdd.py:309: RuntimeWarning: divide by zero encountered in scalar divide
  term2 = T_a


Step 1 alone:       0.049486
Step 2 alone:       0.049486
Sum of individuals: 0.098972
Both together:      0.098972
Additivity holds:   True
